# NBHD-GeoJSON

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
from shapely import Point, MultiPoint, MultiPolygon
import geopandas as gpd
import numpy as np
import pandas as pd
import geopandas as gpd
from libpysal.cg import alpha_shape
import matplotlib.pyplot as plt
import json
from ipywidgets import Widget
import celldega as dega

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.5 when it was built against 1.14.6, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [3]:
import spatialdata as sd
from spatialdata_io import xenium

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [9]:
base_path = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_human_Pancreas_FFPE/main/Landscape_Xenium_V1_human_Pancreas_FFPE_outs_webp/'
zarr_path = 'data/xenium_data/Xenium_V1_human_Pancreas_FFPE_outs.zarr'


In [10]:
cluster = pd.read_parquet(base_path + 'cell_clusters/cluster.parquet')

In [11]:
sdata = sd.read_zarr(zarr_path)
sdata

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, 

SpatialData object, with associated Zarr store: /Users/feni/Documents/celldega/notebooks/data/xenium_data/Xenium_V1_human_Pancreas_FFPE_outs.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 13770, 34155), (5, 6885, 17077), (5, 3442, 8538), (5, 1721, 4269), (5, 860, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (13770, 34155), (6885, 17077), (3442, 8538), (1721, 4269), (860, 2134)
│     └── 'nucleus_labels': DataTree[yx] (13770, 34155), (6885, 17077), (3442, 8538), (1721, 4269), (860, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 11) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (140702, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (140702, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (136531, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (140702, 377)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), 

In [15]:
adata = sdata.tables["table"]
adata.obs.set_index('cell_id', inplace=True)
adata

AnnData object with n_obs × n_vars = 140702 × 377
    obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level', 'nucleus_count', 'cell_labels', 'cluster'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [16]:
adata.obs['cluster'] = cluster

In [17]:
adata.obs.head()

,transcript_counts,control_probe_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,region,z_level,nucleus_count,cell_labels,cluster
cell_id,,,,,,,,,,,,,
aaaadnje-1,37,0,0,0,0,37,44.117658,38.021564,cell_circles,0.0,1.0,1,15
aaacalai-1,60,0,0,0,0,60,66.244221,33.912345,cell_circles,0.0,1.0,2,9
aaacjgil-1,63,0,0,0,0,63,104.491566,52.697346,cell_circles,0.0,1.0,3,15
aaacpcil-1,12,0,0,0,0,12,34.183282,17.520626,cell_circles,0.0,1.0,4,13
aaadhocp-1,143,0,0,0,0,143,149.060787,51.116877,cell_circles,0.0,1.0,5,18


In [18]:
adata.obs.cluster.value_counts()

cluster
1     17949
2     15781
3     14415
4     11840
5      9526
6      8817
7      7220
8      6551
9      5450
10     5396
11     5218
12     3854
13     3813
14     3349
15     3266
16     3021
17     2492
18     1758
19     1722
20     1449
21     1402
22     1189
23     1121
24     1039
25      803
26      739
27      532
28      482
Name: count, dtype: Int64

In [29]:
df['geometry']

0         [446.3266906738281, 1701.3572998046875]
1            [441.3078308105469, 1735.8779296875]
2             [466.0531921386719, 1712.259765625]
3         [430.85809326171875, 1707.464599609375]
4          [476.11114501953125, 1711.08935546875]
                           ...                   
140697         [6082.67578125, 555.1428833007812]
140698      [6106.8994140625, 494.95184326171875]
140699       [6080.9912109375, 626.7421264648438]
140700         [6030.5947265625, 536.50341796875]
140701      [6022.63720703125, 573.7843017578125]
Name: geometry, Length: 140702, dtype: object

In [21]:
# coords = adata.obsm['spatial']
# adata.obs['geometry'] = list(coords)

In [22]:
adata.obs

,transcript_counts,control_probe_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,region,z_level,nucleus_count,cell_labels,cluster,geometry
cell_id,,,,,,,,,,,,,,
aaaadnje-1,37,0,0,0,0,37,44.117658,38.021564,cell_circles,0.0,1.0,1,15,"[446.3266906738281, 1701.3572998046875]"
aaacalai-1,60,0,0,0,0,60,66.244221,33.912345,cell_circles,0.0,1.0,2,9,"[441.3078308105469, 1735.8779296875]"
aaacjgil-1,63,0,0,0,0,63,104.491566,52.697346,cell_circles,0.0,1.0,3,15,"[466.0531921386719, 1712.259765625]"
aaacpcil-1,12,0,0,0,0,12,34.183282,17.520626,cell_circles,0.0,1.0,4,13,"[430.85809326171875, 1707.464599609375]"
aaadhocp-1,143,0,0,0,0,143,149.060787,51.116877,cell_circles,0.0,1.0,5,18,"[476.11114501953125, 1711.08935546875]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
oiloppgp-1,14,0,0,0,0,14,15.082188,15.082188,cell_circles,6.0,1.0,140698,10,"[6082.67578125, 555.1428833007812]"
oilpccne-1,2,0,0,0,0,2,5.734844,5.734844,cell_circles,6.0,1.0,140699,6,"[6106.8994140625, 494.95184326171875]"
oimacfoj-1,11,0,0,0,0,11,13.682344,13.682344,cell_circles,6.0,1.0,140700,10,"[6080.9912109375, 626.7421264648438]"


In [23]:
# import pandas as pd
# import numpy as np

# coords = adata.obsm["spatial"]              # shape: (n_obs, 2)
# df = adata.obs.copy()
# df["geometry"] = list(coords)               # each row gets its [x, y] array
# # meta_cell = pd.DataFrame(df)


In [25]:
# inst_index = adata.obs['cell_id'].values.tolist()
# df_spatial = pd.DataFrame(adata.obsm['spatial'], index=inst_index)

In [26]:
# df_spatial.head()

### Load Data

In [27]:
# meta_cell_ini = pd.read_parquet(base_path + 'cell_metadata.parquet')
# cluster = pd.read_parquet(base_path + 'cell_clusters/cluster.parquet')
# meta_cluster = pd.read_parquet(base_path + 'cell_clusters/meta_cluster.parquet')
# meta_cell = pd.concat([meta_cell_ini, cluster], axis=1)

In [28]:
# meta_cluster.head()

### Calculate Alpha Shape Neighborhoods

In [ ]:
alphas=[20, 50]
gdf_alpha = dega.nbhd.alpha_shape_cell_clusters(adata, cat='cluster', alphas=alphas)
# geojson_alpha = alpha_shape_geojson(gdf_alpha, meta_cluster, inst_alpha=250)

AnnData object with n_obs × n_vars = 140702 × 377
    obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level', 'nucleus_count', 'cell_labels', 'cluster', 'geometry'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'


In [ ]:
gdf_alpha.head()

In [ ]:
gdf_alpha = gdf_alpha[gdf_alpha['inv_alpha'] == 50]

In [ ]:
gdf_alpha.head()

In [ ]:
Widget.close_all()
base_url = base_path.rstrip('/')
landscape = dega.viz.Landscape(
    technology='Xenium',
    height=500,
    base_url = base_url,
    # nbhd=geojson_alpha
    # nbhd_gdf=gdf_alpha,
    nbhd=gdf_alpha,
        
)

landscape

In [12]:
# landscape.nbhd_geojson